# Solution E2 · `predict_proba[:, 1]` is not always the probability you think

**Type**: Find the bug

## Objective

Find out why a model that flags at-risk customers "finds" 94% of the customers (far too many) with a precision of 14%, despite an excellent AUC: the columns of `predict_proba` are
in **alphabetical** order of the classes, and with text labels column 1 is not the one the colleague thinks. Fix it and write a robust function.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
y_txt = np.where(df["defaulted"] == 1, "defaulted", "regular")
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], y_txt, test_size=0.3, stratify=y_txt, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)

## Commented solution

In [ ]:
p = model.predict_proba(X_te)[:, 1]                                         # "default probability"
auc_bug = roc_auc_score(y_te, p)
flagged = p >= 0.3                                                          # flag as defaulter whoever has p >= 0.3
pred = np.where(flagged, "defaulted", "regular")
bug_flagged_share = flagged.mean()
bug_precision = precision_score(y_te, pred, pos_label="defaulted")
print(f"AUC {auc_bug:.3f} | customers flagged {bug_flagged_share:.1%} | precision {bug_precision:.3f}")

**Where the bug is.** `predict_proba` orders the columns like `classes_`, that is in **alphabetical order**: `["defaulted", "regular"]`. Column 1 is the probability of `"regular"`, not of defaulted. The colleague flags whoever has
`P(regular) >= 0.3`, that is almost everyone. The AUC comes out right because `roc_auc_score` with text labels takes the last class in alphabetical order (`"regular"`) as positive and column 1 is exactly its probability: two
errors that cancel in the metric and not in the decision.

In [ ]:
classes = list(model.classes_)
defaulted_index = classes.index("defaulted")
print("classes_:", classes, "-> index of 'defaulted':", defaulted_index)

In [ ]:
p_def = model.predict_proba(X_te)[:, defaulted_index]
flagged = p_def >= 0.3
pred = np.where(flagged, "defaulted", "regular")
ok_flagged_share = flagged.mean()
ok_precision = precision_score(y_te, pred, pos_label="defaulted")
ok_recall = recall_score(y_te, pred, pos_label="defaulted")
print(f"customers flagged {ok_flagged_share:.1%} | precision {ok_precision:.3f} | recall {ok_recall:.3f}")

In [ ]:
def class_proba(model, X, cls):
    """Probability of the requested class, looked up by name (never by position)."""
    return model.predict_proba(X)[:, list(model.classes_).index(cls)]

## Check

In [ ]:
with verify():
    check(auc_bug > 0.78, "the colleague's AUC is good: the bug hides", "don't modify the colleague's code")
    check(bug_flagged_share > 0.9 and bug_precision < 0.2, "the colleague flags over 90% of the customers with a precision below 20%", "don't modify the colleague's code")
    check(classes == ["defaulted", "regular"] and defaulted_index == 0, "the classes are in alphabetical order: 'defaulted' is column 0", "model.classes_")
    check(close(p_def, model.predict_proba(X_te)[:, 0]), "the default probability is column 0", "use defaulted_index")
    check(ok_flagged_share < 0.25 and ok_precision > 0.5 and ok_recall > 0.5, "with the right column: about 18% flagged, precision and recall above 0.5", "threshold 0.3 on p_def, positive class 'defaulted'")
    m1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, (y_tr == "defaulted").astype(int))
    m2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, np.where(y_tr == "defaulted", "yes", "no"))
    check(close(class_proba(m1, X_te, 1), m1.predict_proba(X_te)[:, 1]) and close(class_proba(m2, X_te, "yes"), m2.predict_proba(X_te)[:, 1]) and close(class_proba(m2, X_te, "yes"), 1 - class_proba(m2, X_te, "no")),
          "class_proba works with numeric and text labels, in any order", "look up the column with list(model.classes_).index(cls)")
done()

## What you learned

- **The columns of `predict_proba` follow `classes_`, which is sorted.** With a 0/1 target column 1 is the positive class; with text labels (`"no"/"yes"`, `"defaulted"/"regular"`) it can be the opposite one. Never trust
  the position: look it up by name.
- One bug can be hidden by another: the AUC computed on the strings comes out **right by chance** (positive = last alphabetical class, like column 1) and the check everybody does first does not see it. The symptom
  is elsewhere: here 94% of the customers flagged and a precision of 14% (the prevalence, like flagging everyone).
- For the metrics that depend on the positive class (`precision_score`, `recall_score`, `f1_score`) with text labels **always** pass `pos_label`, otherwise scikit-learn uses the one it likes (and with more than two values it raises an error).
- The remedy is simple: convert the target to 0/1 with the positive class explicit (`y == "defaulted"`) at the start of the data pipeline, or use a function like `class_proba`.

## Common pitfall

Writing `predict_proba(X)[:, 1]` as if it were an always-valid idiom. It is valid with a 0/1 target and `classes_ = [0, 1]`; with booleans, text or classes not ordered the way you expect, it is not. Check `classes_` at least once per project.

## Interview question

<details><summary>How do you make sure you use the right column of predict_proba?</summary>

I encode the target as 0/1 with the class of interest as 1 at the start, and check `model.classes_`. If the target has text labels I look up the column by name and not by position. In the automated tests I verify
that the mean predicted probability is close to the frequency of the positive class: it is a very simple check that immediately catches swapped columns.

</details>